In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict

# Configuration
result_dir = Path("/local/scratch/zli2255/workspace/MergeExpert/eval_results/reeval/tool")

# Find all score directories
score_dirs = list(result_dir.rglob("score"))
print(f"Found {len(score_dirs)} score directories")

# Group by model name
model_runs = defaultdict(list)

for score_dir in score_dirs:
    model_path = score_dir.parent
    model_name = model_path.name
    
    # Extract run number from path
    run_num = None
    for part in model_path.parts:
        if part.startswith('run_'):
            try:
                run_num = int(part.split('_')[1])
            except:
                pass
    
    # Check if all required CSV files exist
    live_csv = score_dir / "data_live.csv"
    non_live_csv = score_dir / "data_non_live.csv"
    multi_turn_csv = score_dir / "data_multi_turn.csv"
    
    if live_csv.exists() and non_live_csv.exists() and multi_turn_csv.exists():
        model_runs[model_name].append({
            'path': score_dir,
            'run': run_num,
        })

print(f"Found {len(model_runs)} unique models")

# Process each model
results = []

for model_name, runs in sorted(model_runs.items()):
    print(f"Processing {model_name} ({len(runs)} run(s))...")
    
    # Collect metrics from all runs
    live_accs = []
    non_live_accs = []
    multi_turn_accs = []
    
    for run_info in runs:
        score_dir = run_info['path']
        
        try:
            # Read the three CSV files
            live_df = pd.read_csv(score_dir / "data_live.csv")
            non_live_df = pd.read_csv(score_dir / "data_non_live.csv")
            multi_turn_df = pd.read_csv(score_dir / "data_multi_turn.csv")
            
            # Extract accuracy values (remove % sign and convert to float)
            live_acc = float(live_df.iloc[0]['Live Overall Acc'].rstrip('%'))
            non_live_acc = float(non_live_df.iloc[0]['Non-Live Overall Acc'].rstrip('%'))
            multi_turn_acc = float(multi_turn_df.iloc[0]['Multi Turn Overall Acc'].rstrip('%'))
            
            live_accs.append(live_acc)
            non_live_accs.append(non_live_acc)
            multi_turn_accs.append(multi_turn_acc)
            
            print(f"  Run {run_info['run']}: live={live_acc:.2f}, non_live={non_live_acc:.2f}, multi_turn={multi_turn_acc:.2f}")
            
        except Exception as e:
            print(f"  Error processing run {run_info['run']}: {e}")
            continue
    
    if live_accs:
        # Calculate mean values
        live_mean = round(np.mean(live_accs), 2)
        non_live_mean = round(np.mean(non_live_accs), 2)
        multi_turn_mean = round(np.mean(multi_turn_accs), 2)
        avg_accuracy = round(np.mean([live_mean, non_live_mean, multi_turn_mean]), 2)
        
        results.append({
            'model': model_name,
            'num_runs': len(live_accs),
            'live_acc': live_mean,
            'non_live_acc': non_live_mean,
            'multi_turn_acc': multi_turn_mean,
            'avg_accuracy': avg_accuracy,
        })

# Check if we have results
if not results:
    print("\n❌ Error: No valid results found!")
    print(f"Please check if the directory exists and contains valid data:")
    print(f"  {result_dir}")
    print(f"\nExpected structure:")
    print(f"  <model_name>/score/data_live.csv")
    print(f"  <model_name>/score/data_non_live.csv")
    print(f"  <model_name>/score/data_multi_turn.csv")
else:
    # Create DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('avg_accuracy', ascending=False)

    # Save to CSV
    output_path = result_dir / "tool_summary.csv"
    results_df.to_csv(output_path, index=False)
    print(f"\n✓ Results saved to: {output_path}")

    # Display
    display(results_df)


Found 43 score directories
Found 9 unique models
Processing tool_merge_dare_ties_3B (4 run(s))...
  Run 3: live=68.69, non_live=81.85, multi_turn=3.12
  Run 4: live=68.76, non_live=81.67, multi_turn=3.50
  Run 1: live=68.62, non_live=81.40, multi_turn=3.62
  Run 5: live=68.69, non_live=81.81, multi_turn=2.88
Processing tool_merge_simag_cos_percentile10_1.5B (5 run(s))...
  Run 3: live=65.66, non_live=72.96, multi_turn=2.88
  Run 2: live=65.66, non_live=72.81, multi_turn=2.88
  Run 4: live=65.51, non_live=72.73, multi_turn=2.63
  Run 1: live=65.73, non_live=72.75, multi_turn=2.75
  Run 5: live=65.58, non_live=72.73, multi_turn=2.88
Processing tool_merge_simag_cos_percentile20_1.5B (5 run(s))...
  Run 3: live=65.80, non_live=72.98, multi_turn=2.63
  Run 2: live=65.36, non_live=73.10, multi_turn=2.88
  Run 4: live=65.28, non_live=72.88, multi_turn=2.75
  Run 1: live=65.73, non_live=73.10, multi_turn=2.63
  Run 5: live=65.66, non_live=72.94, multi_turn=2.63
Processing tool_merge_simag_cos_

,model,num_runs,live_acc,non_live_acc,multi_turn_acc,avg_accuracy
8,tool_mix_3B,5,68.47,82.49,4.40,51.79
0,tool_merge_dare_ties_3B,4,68.69,81.68,3.28,51.22
5,tool_merge_simag_cos_percentile40_3B,5,68.63,81.60,3.38,51.20
7,tool_merge_simag_cos_percentile50_3B,4,68.08,81.17,0.00,49.75
6,tool_merge_simag_cos_percentile50_1.5B,5,62.68,75.46,3.20,47.11
2,tool_merge_simag_cos_percentile20_1.5B,5,65.57,73.00,2.70,47.09
1,tool_merge_simag_cos_percentile10_1.5B,5,65.63,72.80,2.80,47.08
3,tool_merge_simag_cos_percentile30_1.5B,5,64.89,73.16,2.58,46.88
4,tool_merge_simag_cos_percentile40_1.5B,5,63.50,74.09,3.00,46.86
